# General magnetic domains: imbalance, morphology and ordering

Start with the experimental **constrained dipolar model** below. It generates
fully magnetized +1/-1 domains while constraining net magnetization and measuring
the requested size. The later procedural baseline and its catalogue remain
available for comparison. Both models generate scalar textures, not vector LLG
trajectories or topological skyrmion winding.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/scattering_calculator").is_dir())
if str(root / "src") not in sys.path: sys.path.insert(0, str(root / "src"))
from scattering_calculator.sample_generator import create_domain_pattern, create_constrained_domain_pattern
from scattering_calculator.sample_generator.domain_catalogue import measure_domain_pattern


## Experimental constrained dipolar model

This is the first literature-inspired relaxation backend, separate from the
procedural examples below. It includes a local double well, interfacial stiffness,
long-wavelength dipolar competition, directional stiffness and optional quenched
pinning. [Model and limitations](../docs/constrained_magnetic_domains.md).

`target_magnetization` sets **occupied area**, not amplitude: all output pixels
are +1 or -1. The area constraint acts at every relaxation step. The preferred
repeat is adjusted against the measured radial Fourier peak, with bounded work.
A reported convergence or size target can fail; inspect the metadata.


In [ ]:
constrained_shape = (128, 128)
constrained_parameters = dict(
    target_magnetization=0.0,  # M/Ms: 0 = half up, half down; +0.6 = 80% up
    characteristic_length=16.0,  # target full repeat/spacing, in pixels
    alignment=0.0,  # try 8 for aligned stripes
    direction=0.0,  # stripe-axis angle in radians
    disorder=0.0, disorder_length=1.0,
    initial_state="random",  # random, stripes, hexagonal
    initial_order=0.0,  # blend ordered seed into random field
    wall_stiffness=2.0, n_steps=400, time_step=0.2,
    length_tolerance=0.15, max_length_iterations=5, seed=7,
)
constrained_mz, constrained_meta = create_constrained_domain_pattern(
    constrained_shape, **constrained_parameters)
print({key: constrained_meta[key] for key in (
    "mean_mz", "up_fraction_measured", "measured_length_px", "length_target_met",
    "converged", "residual", "steps_completed", "nematic_order", "elapsed_seconds")})
assert set(np.unique(constrained_mz)) <= {-1., 1.}
assert all(row["up_pixels"] == constrained_meta["target_up_pixels"] for row in constrained_meta["trace"])
plt.figure(figsize=(5, 5))
plt.imshow(constrained_mz, cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
plt.title("Constrained dipolar relaxation; fully magnetized domains")
plt.colorbar(label="mz"); plt.show()


### Measured model examples

The saved comparison includes labyrinths, aligned stripes, minority bubbles,
ordered bubble and stripe preparations, and a pinned labyrinth. It records
achieved size, elapsed time and whether relaxation reached the residual
criterion. A finite-step pattern may remain metastable or still be relaxing.
These examples are not a benchmark against full micromagnetics.


In [ ]:
trial_dir = root / "docs/assets/constrained_domains"
if (trial_dir / "measurements.json").exists():
    trial_results = json.loads((trial_dir / "measurements.json").read_text())
    print(f"{len(trial_results['records'])} measured model trials")
    plt.figure(figsize=(14, 10))
    plt.imshow(plt.imread(trial_dir / "morphologies.png"))
    plt.axis("off"); plt.show()
else:
    print("Generate the measured trial with: PYTHONPATH=src python scripts/catalogue_constrained_domains.py")


## Editable controls

`target_mean` constrains the evolving continuous field; `up_fraction` constrains
the final sign area fraction. Domains are +1 and -1; the fraction changes
occupied area, never their magnetization strength. They are different quantities. Use `None` for
`up_fraction` to observe the occupancy selected by evolution. Ordered positive
bubble lattices can be seeded with `lattice_order=1, up_fraction=0.15`.
Lengths below are pixels; `MagneticPatternConfig` accepts metres.


In [ ]:
parameters = dict(period=16., up_fraction=.3, target_mean=None,
                  lattice_order=0., lattice_angle=0., k0=.7, eps=.8,
                  quadratic_coefficient=0., field_bias=0., n_steps=80,
                  noise_amp=0., sigma=.7, domain_conversion="hard", seed=11, use_gpu=False)
shape = (128, 128)
mz, metadata = create_domain_pattern(shape, **parameters)
metrics = measure_domain_pattern(mz)
print(metrics)
plt.figure(figsize=(5,5))
plt.imshow(mz, cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
plt.colorbar(label="mz"); plt.title("Scalar magnetic domains"); plt.show()


## Same model, different shapes and fractions

The first row uses random initial conditions; the second uses hexagonal seeds.
Quantile calibration fixes occupancy, while the model and seed determine shape.


In [ ]:
fractions = [.15, .3, .5, .7, .85]
orders = [0., 1.]
fig, axes = plt.subplots(len(orders), len(fractions), figsize=(14,6), squeeze=False, constrained_layout=True)
for row, order in enumerate(orders):
    for col, fraction in enumerate(fractions):
        controls = dict(parameters, up_fraction=fraction, lattice_order=order)
        pattern, meta = create_domain_pattern(shape, **controls)
        axes[row,col].imshow(pattern, cmap="RdBu_r", vmin=-1, vmax=1)
        axes[row,col].set_title(f"order={order:g}, f+={meta['up_fraction_measured']:.2f}")
        axes[row,col].set_axis_off()
plt.show()


## Natural occupancy from evolution

No quantile correction is applied in this comparison. The printed mean `mz`
is the area-weighted mean of fully magnetized domains: `mean_mz = 2*f_up - 1`.


In [ ]:
means = [-.8, -.4, 0., .4, .8]
fig, axes = plt.subplots(1, len(means), figsize=(14,3), squeeze=False, constrained_layout=True)
for ax, mean in zip(axes.ravel(), means):
    pattern, meta = create_domain_pattern(shape, **dict(parameters, target_mean=mean, up_fraction=None))
    ax.imshow(pattern, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(f"mean control={mean:g}\nf+={meta['up_fraction_measured']:.2f}, <mz>={meta['mean_mz']:.2f}")
    ax.set_axis_off()
plt.show()


## Labyrinthine stripes from Tutorial 03

The earlier examples emphasized occupancy and ordered bubbles. Tutorial 03 also
varies the native spectral wavelength and uses the legacy explicit evolution.
Here the same general function uses `rescale_mode="native"`,
`stable_evolution=False`, `target_mean=0`, and no occupancy correction.
The period follows `k0`; the `period` argument is not applied in native mode.
Domains remain +1/-1. Lower `k0` produces broader, smoother labyrinthine stripes.


In [ ]:
stripe_k0 = [.2, .7]
stripe_eps = [.1, .6, 1.2]
fig, axes = plt.subplots(len(stripe_k0), len(stripe_eps), figsize=(12,8), squeeze=False, constrained_layout=True)
for row, k0 in enumerate(stripe_k0):
    for col, eps in enumerate(stripe_eps):
        pattern, meta = create_domain_pattern(
            shape, period=16, k0=k0, eps=eps, target_mean=0., up_fraction=None,
            lattice_order=0., stable_evolution=False, rescale_mode="native",
            sigma=None, domain_conversion="hard", max_hole_area=9,
            n_steps=80, noise_amp=0., seed=7, use_gpu=False)
        metrics = measure_domain_pattern(pattern)
        axes[row,col].imshow(pattern, cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
        axes[row,col].set_title(f"k0={k0:g}, eps={eps:g}; f+={metrics['up_fraction']:.2f}")
        axes[row,col].set_axis_off()
plt.show()


## Inspect the measured catalogue

Every record includes parameters and observations; shape labels are empirical.
Border-clipped circles can be classified as stripe-like. Sixfold order is a
center-neighbor statistic, not a skyrmion topological invariant.


In [ ]:
catalogue = json.loads((root / "docs/assets/domain_catalogue/catalogue.json").read_text())
records = catalogue["records"]
print(f"{len(records)} measured samples; seeds={catalogue['seeds']}")
for panel in dict.fromkeys(record["panel"] for record in records):
    plt.figure(figsize=(14,9))
    plt.imshow(plt.imread(root / "docs/assets/domain_catalogue" / f"{panel}.png"))
    plt.axis("off"); plt.show()
